# 이커머스 Uplift 분석 — A파트(가격구간, H1) 모델링 노트북

**프로젝트**: 친환경 신선식품 이커머스 FRESH.DATA — 구독 전환 Uplift 기반 프로모션 타겟팅
**담당(A파트)**: H1(가격구간) 가설 검증 + 팀 전체 백테스팅·최종 타겟팅 리스트 산출
**작성일**: 2026-08-24

## 배경
쿠폰 노출(Treatment) 컬럼이 원본 데이터에 없어, **Treatment=구독여부**로 재정의하고 T-learner 구조로
"구독을 유도하면 실제로 재구매가 늘어날 사람(Persuadable)"을 찾는 것이 프로젝트의 핵심 질문이다.
Y=14일 내 재구매여부, Train index_date=2025-09-16, Holdout index_date=2025-10-26(원본 데이터 자체의
11/10~14 결측 구간을 피하기 위해 8/24 11/2에서 재설정, 상세는 `진행_기록.md` 20절 참고).

## 이 노트북의 구성 (10-Run 계획 중 A파트 담당 run)
0. 전처리 파이프라인 — Train/Holdout 스냅샷 생성
1. **run01b** — L1 정규화 로지스틱 T-learner (대표 모델)
2. **run08b** — RandomForest T-learner (GridSearchCV 재튜닝)
3. **run06** — 매출 관점 Uplift (Y=14일 매출액)
4. **run07** — 이탈위험군별 Uplift
5. **run09** — 백테스팅 (A+B+C+D 4파트 개인별 Uplift 점수 교차검증)
6. **run10** — 전체 결과 리더보드
7. **run11** — 최종 타겟팅 리스트 산출
8. 결론 요약

> ⚠️ run09/11은 B/C/D(팀원)가 제공한 `uplift_scores_part{B,C,D}*.csv`에 의존한다. 이 노트북 실행 시점에
> 그 파일들이 어느 Holdout 기준으로 계산됐는지에 따라 5·7번 섹션의 구체적 수치가 달라질 수 있다
> (상세: `파일_인덱스.md`, `홀드아웃_재생성_재검증_프롬프트_BCD.md`).

---
## 0. 전처리 파이프라인

원본 3테이블(Sales/Member/Product)을 병합한 `merged_master.csv`에서 시작해, 회원×index_date 단일
스냅샷(Train/Holdout)을 만든다. RFM·가격구간(Train 기준 분위수)·region_tier·제철상품 태깅·이상거래
플래그·구독여부 재분류(Treatment) 등 핵심 파생변수를 이 단계에서 전부 생성한다.

In [1]:
# -*- coding: utf-8 -*-
"""
전처리 파이프라인 v1
- 8-Step 계획(CLAUDE.md) + A/B/C/D 통합 인사이트(전처리_통합_인사이트_ABCD.md) 반영
- 분석단위: 단일 스냅샷(회원당 1행) x Train/Holdout 2개
  Train  index_date = 2025-09-16, Y = 9/17~9/30 재구매여부(14일)
  Holdout index_date = 2025-10-26, Y = 10/27~11/9 재구매여부(14일)
  (8/24 변경: 원래 11/2였으나, 원본 Sales_Data 자체에 11/10~14 5일 연속 결측이 있어
   14일 라벨 윈도우 안에 결측이 절반 가까이 걸림 -> Y가 과소추정될 위험 발견(팀원 EDA로 확인).
   원본에 결측이 없는 10/3·10/11~12·11/10~14를 모두 피하는 구간으로 재설정.)
- 이번 버전 범위에 포함 O: 지역 정규화/region_tier, 계절(제철) 태깅, RFM, 가격구간,
  적립금 이력, 연령x성별 세그먼트, 구독여부 재분류(Treatment), Y(14일), 이상치 플래그
- 이번 버전 범위에서 제외(TODO, 핵심 모델링에 필수 아님): 등록카드 3분류, 상품중량 정규식 정제,
  주문시간 "XX:60" 보정 -> 시간대 분석 필요해지면 추가
"""
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
RAW_PATH = BASE + "merged_master.csv"

TRAIN_INDEX_DATE = pd.Timestamp("2025-09-16")
HOLDOUT_INDEX_DATE = pd.Timestamp("2025-10-26")  # 8/24: 11/10~14 원본결측 회피 위해 11/2->10/26 변경
N_DAYS = 14
DATA_MAX_DATE = pd.Timestamp("2025-11-16")

# ---------------------------------------------------------------------------
# Step 0: 로드 & 무결성 재확인
# ---------------------------------------------------------------------------
df = pd.read_csv(RAW_PATH, encoding="utf-8-sig")
assert df.shape == (668111, 36), f"[Step0] shape 불일치: {df.shape}"
assert df["회원번호"].isna().sum() == 0
assert df["제품번호"].isna().sum() == 0
df["주문일시_dt"] = pd.to_datetime(df["주문일시"])
assert df["주문일시_dt"].min() == pd.Timestamp("2025-01-11")
assert df["주문일시_dt"].max() == pd.Timestamp("2025-11-16")
print("[Step0] OK - shape/키/날짜범위 검증 통과")

# ---------------------------------------------------------------------------
# Step 1: 컬럼 정제 - 주소지 정규화, region_key, region_tier (D파트 반영)
# ---------------------------------------------------------------------------
ADDR_NORMALIZE = {"경기": "경기도", "광주": "광주광역시", "강원": "강원도", "서울": "서울특별시"}
df["주소지_정규화"] = df["주소지"].replace(ADDR_NORMALIZE)

# 세종시는 세부주소지를 동/로 단위로 세분화하지 않고 하나로 롤업 (D 3.2 제안)
df["세부주소지_정제"] = df["세부주소지"]
sejong_mask = df["주소지_정규화"] == "세종특별자치시"
df.loc[sejong_mask, "세부주소지_정제"] = "세종시"

df["region_key"] = df["주소지_정규화"].astype(str) + "_" + df["세부주소지_정제"].astype(str)

TIER1 = {"서울특별시", "경기도", "인천광역시"}
TIER2 = {"부산광역시", "대구광역시", "대전광역시", "광주광역시", "울산광역시", "세종특별자치시"}
TIER3 = {"강원도", "충청남도", "충청북도", "전라남도", "전라북도", "경상남도", "경상북도", "제주특별자치도"}


def to_tier(addr):
    if addr in TIER1:
        return "Tier1_새벽배송가능권(가정)"
    if addr in TIER2:
        return "Tier2_익일배송표준권(가정)"
    if addr in TIER3:
        return "Tier3_배송취약권(가정)"
    return "미상"


df["region_tier"] = df["주소지_정규화"].map(to_tier)
df["is_dawn_delivery_zone"] = (df["region_tier"] == "Tier1_새벽배송가능권(가정)").astype(int)
print("[Step1] 지역 정규화/region_key/region_tier 생성 완료")
print(df["region_tier"].value_counts(dropna=False).to_dict())

# 취소/배송 관련 플래그
df["is_cancelled"] = (df["주문취소여부"] == "주문취소").astype(int)
df["is_cancelled_no_delivery"] = df["is_cancelled"]  # D: 배송일 결측=취소와 100% 일치, 동일 플래그로 취급
df["delivery_leadtime_days"] = (
    pd.to_datetime(df["배송완료일"]) - df["주문일시_dt"]
).dt.days  # 취소건은 NaN 유지 (구조적 결측)

# 요일/주말여부 (C파트)
df["요일"] = df["주문일시_dt"].dt.day_name()
df["주말여부"] = df["주문일시_dt"].dt.dayofweek.isin([5, 6]).astype(int)

# ---------------------------------------------------------------------------
# Step 1b: 명절선물세트여부 (C파트) - 식품 계열 + 키워드
# ---------------------------------------------------------------------------
GIFT_KEYWORDS = r"선물모음|선물용|선물세트|세트"
FOOD_CATEGORIES = set(df.loc[~df["물품대분류"].astype(str).str.contains("생활|화장|잡화|리빙", na=False), "물품대분류"].unique())
df["명절선물세트여부"] = (
    df["물품명"].astype(str).str.contains(GIFT_KEYWORDS, regex=True, na=False)
    & df["물품대분류"].isin(FOOD_CATEGORIES)
).astype(int)
print(f"[Step1b] 명절선물세트 태깅 {df['명절선물세트여부'].sum()}건")

# ---------------------------------------------------------------------------
# Step 1c: 자체 공휴일 캘린더 보정 + 연휴전후구간 (C파트 3-1)
# ---------------------------------------------------------------------------
HOLIDAY_DATES = pd.to_datetime(
    [
        "2025-01-29", "2025-01-30", "2025-01-31",  # 설
        "2025-03-01",                                # 삼일절 (보정 추가)
        "2025-05-05", "2025-05-06", "2025-05-08",   # 어린이날/부처님오신날 대체
        "2025-06-06",                                 # 현충일
        "2025-08-15",                                 # 광복절
        "2025-10-03", "2025-10-04", "2025-10-05", "2025-10-06",  # 개천절~추석연휴~추석당일
        "2025-10-07", "2025-10-08",                                 # 추석연휴~대체공휴일
        "2025-10-09",                                                # 한글날
    ]
)
holiday_set = set(HOLIDAY_DATES)
pre_holiday = set().union(*[set(HOLIDAY_DATES - pd.Timedelta(days=d)) for d in (1, 2, 3)])
post_holiday = set().union(*[set(HOLIDAY_DATES + pd.Timedelta(days=d)) for d in (1, 2, 3)])


def to_holiday_period(d):
    if d in holiday_set:
        return "연휴중"
    if d in pre_holiday:
        return "연휴직전"
    if d in post_holiday:
        return "연휴직후"
    return "평시"


df["연휴전후구간"] = df["주문일시_dt"].map(to_holiday_period)
print("[Step1c]", df["연휴전후구간"].value_counts().to_dict())

# ---------------------------------------------------------------------------
# Step 2a: 제철여부 (C파트 3-2) - 물품중분류별 월별 판매비중 top3 >=65%
# ---------------------------------------------------------------------------
valid = df[df["is_cancelled"] == 0].copy()
monthly_qty = valid.groupby(["물품중분류", "주문년월"])["구매수량"].sum().reset_index()
total_qty = monthly_qty.groupby("물품중분류")["구매수량"].transform("sum")
monthly_qty["비중"] = monthly_qty["구매수량"] / total_qty
item_total_count = valid.groupby("물품중분류").size()

seasonal_months = {}
for item, g in monthly_qty.groupby("물품중분류"):
    top3 = g.nlargest(3, "비중")
    concentration = top3["비중"].sum()
    if item_total_count.get(item, 0) < 200:
        seasonal_months[item] = None  # 표본부족 -> 상시 취급(외부데이터 보완 필요, TODO)
    elif concentration >= 0.65:
        seasonal_months[item] = set(top3["주문년월"])
    else:
        seasonal_months[item] = None  # 상시 품목

n_seasonal_items = sum(1 for v in seasonal_months.values() if v is not None)
print(f"[Step2a] 계절성 품목 {n_seasonal_items}/{len(seasonal_months)}개 (임계값 65%, 표본<200건은 상시 처리)")


def is_in_season(row):
    months = seasonal_months.get(row["물품중분류"])
    if months is None:
        return 0
    return int(row["주문년월"] in months)


df["제철상품구매여부"] = df.apply(is_in_season, axis=1)

# ---------------------------------------------------------------------------
# Step 2b: 이상치 플래그 (Step3 항목이지만 스냅샷 계산 전에 필요해 미리 생성)
#   - amount_outlier_flag: TRAIN 기간 basket_amt IQR 기준(정보 누수 방지, A파트)
# ---------------------------------------------------------------------------
basket_all = (
    df[df["is_cancelled"] == 0]
    .groupby(["회원번호", "주문일시_dt"], as_index=False)
    .agg(
        basket_amt=("구매금액", "sum"),
        basket_item_count=("구매금액", "size"),
        reward_used=("사용 적립금", lambda s: int((s > 0).any())),
        reward_amt=("사용 적립금", "sum"),
    )
)
train_baskets_for_iqr = basket_all[basket_all["주문일시_dt"] <= TRAIN_INDEX_DATE]
q1, q3 = train_baskets_for_iqr["basket_amt"].quantile([0.25, 0.75])
iqr = q3 - q1
iqr_lo, iqr_hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
basket_all["amount_outlier_flag"] = (
    (basket_all["basket_amt"] < iqr_lo) | (basket_all["basket_amt"] > iqr_hi)
).astype(int)
print(f"[Step2b] basket_amt IQR(Train기준) = [{iqr_lo:.0f}, {iqr_hi:.0f}], 이상치 {basket_all['amount_outlier_flag'].mean()*100:.1f}%")

print("스크립트 1부(Step0~2b) 완료")

# ---------------------------------------------------------------------------
# Step 2c: 회원 정적 속성 테이블 (연령대/성별/지역 등, B/D파트)
# ---------------------------------------------------------------------------
def age_band_h4(age):
    if age < 30:
        return "<30"
    if age < 40:
        return "30s"
    if age < 50:
        return "40s"
    if age < 60:
        return "50s"
    return "60+"


member_static = df.groupby("회원번호").first().reset_index()[
    ["회원번호", "나이", "성별", "결혼", "주소지_정규화", "region_key", "region_tier", "is_dawn_delivery_zone", "구독여부"]
]
member_static["age_band_h4"] = member_static["나이"].map(age_band_h4)
member_static["gender_h4"] = member_static["성별"].map({"여": "F", "남": "M"}).fillna("Unknown")
member_static["age_gender_segment"] = member_static["age_band_h4"] + "_" + member_static["gender_h4"]
print("[Step2c] 회원 정적 속성 테이블:", member_static.shape)

# 제철구매비율/명절선물세트구매여부용 원본(취소 제외) 행 데이터
item_valid = df[df["is_cancelled"] == 0][
    ["회원번호", "주문일시_dt", "제철상품구매여부", "명절선물세트여부"]
]


# ---------------------------------------------------------------------------
# Step 2d: 스냅샷 빌더 (Train/Holdout 공용 함수)
# ---------------------------------------------------------------------------
def build_snapshot(index_date, label_start, label_end, price_bins=None):
    pre_b = basket_all[basket_all["주문일시_dt"] <= index_date].copy()
    pre_i = item_valid[item_valid["주문일시_dt"] <= index_date]

    agg = pre_b.groupby("회원번호").agg(
        frequency=("basket_amt", "size"),
        monetary=("basket_amt", "sum"),
        aov=("basket_amt", "mean"),
        last_date=("주문일시_dt", "max"),
        reward_ever_used=("reward_used", "max"),
        reward_usage_rate=("reward_used", "mean"),
    ).reset_index()
    agg["recency_days"] = (index_date - agg["last_date"]).dt.days

    # 재구매 간격 규칙성(CV) - subscription 재분류용 보조피처
    pre_b_sorted = pre_b.sort_values(["회원번호", "주문일시_dt"])
    pre_b_sorted["interval"] = pre_b_sorted.groupby("회원번호")["주문일시_dt"].diff().dt.days
    cv = pre_b_sorted.groupby("회원번호")["interval"].agg(lambda s: s.std() / s.mean() if s.mean() else np.nan)
    agg["regularity_cv"] = agg["회원번호"].map(cv)
    median_cv = agg["regularity_cv"].median()
    agg["regularity_cv"] = agg["regularity_cv"].fillna(median_cv)

    seasonal_share = pre_i.groupby("회원번호")["제철상품구매여부"].mean().rename("preperiod_제철구매비율")
    gift_flag = pre_i.groupby("회원번호")["명절선물세트여부"].max().rename("preperiod_명절선물세트구매여부")
    agg = agg.merge(seasonal_share, on="회원번호", how="left").merge(gift_flag, on="회원번호", how="left")
    agg["preperiod_제철구매비율"] = agg["preperiod_제철구매비율"].fillna(0)
    agg["preperiod_명절선물세트구매여부"] = agg["preperiod_명절선물세트구매여부"].fillna(0)

    # 이상거래계정 플래그: 이 스냅샷 preperiod frequency 상위 1%
    p99 = agg["frequency"].quantile(0.99)
    agg["abnormal_account_flag"] = (agg["frequency"] >= p99).astype(int)

    snap = agg.merge(member_static, on="회원번호", how="left")

    # Y: index_date 초과 ~ label_end 이내 유효 basket 존재 여부
    future = basket_all[(basket_all["주문일시_dt"] > index_date) & (basket_all["주문일시_dt"] <= label_end)]
    repurchase_ids = set(future["회원번호"].unique())
    snap["y_repurchase_14d"] = snap["회원번호"].isin(repurchase_ids).astype(int)
    future_revenue = future.groupby("회원번호")["basket_amt"].sum().rename("y_revenue_14d")
    snap = snap.merge(future_revenue, on="회원번호", how="left")
    snap["y_revenue_14d"] = snap["y_revenue_14d"].fillna(0.0)  # run06: 매출 관점 Uplift용 Y(14일 내 재구매 금액, 미구매시 0)
    snap["label_observable_flag"] = int(label_end <= DATA_MAX_DATE)

    # 가격구간 (Train 기준 분위수 경계 고정, A파트)
    if price_bins is None:
        _, price_bins = pd.qcut(snap["aov"], 4, retbins=True, duplicates="drop")
        price_bins = price_bins.copy()
        price_bins[0], price_bins[-1] = -np.inf, np.inf
    snap["가격구간"] = pd.cut(snap["aov"], bins=price_bins, labels=["Q1_저가", "Q2", "Q3", "Q4_고가"])

    # Treatment(구독여부) 원본 + 재분류
    snap["treatment_source"] = np.select(
        [snap["구독여부"] == True, snap["구독여부"] == False],
        ["original_true", "original_false"],
        default="unresolved",
    )

    feat_cols = ["frequency", "monetary", "recency_days", "reward_usage_rate", "regularity_cv", "나이"]
    snap["log_frequency"] = np.log1p(snap["frequency"])
    snap["log_monetary"] = np.log1p(snap["monetary"])
    snap["log_recency"] = np.log1p(snap["recency_days"])
    model_feats = ["log_frequency", "log_monetary", "log_recency", "reward_usage_rate", "regularity_cv", "나이"]

    known = snap[snap["treatment_source"] != "unresolved"]
    unknown = snap[snap["treatment_source"] == "unresolved"]
    prob = pd.Series(index=snap.index, dtype=float)
    if len(unknown) > 0 and len(known) > 0:
        clf = LogisticRegression(class_weight="balanced", max_iter=1000)
        clf.fit(known[model_feats], known["구독여부"] == True)
        prob.loc[known.index] = clf.predict_proba(known[model_feats])[:, 1]
        prob.loc[unknown.index] = clf.predict_proba(unknown[model_feats])[:, 1]
    snap["구독_추정확률"] = prob  # 참고용. unresolved 구간은 0.40~0.58 근처로 사실상 판별력 없음(팀 검토 완료, 8/23)

    # 8/23 결정: 행동기반 강제분류(0.5 threshold) 대신 unresolved는 NaN으로 남기고
    # 메인 Uplift 분석 표본에서 제외한다. include_in_uplift_model로 필터링.
    snap["treatment_h4"] = np.select(
        [snap["treatment_source"] == "original_true", snap["treatment_source"] == "original_false"],
        [1, 0],
        default=np.nan,
    )
    snap["include_in_uplift_model"] = (snap["treatment_source"] != "unresolved").astype(int)

    return snap, price_bins


snap_train, PRICE_BINS = build_snapshot(
    TRAIN_INDEX_DATE, TRAIN_INDEX_DATE + pd.Timedelta(days=1), TRAIN_INDEX_DATE + pd.Timedelta(days=N_DAYS)
)
snap_holdout, _ = build_snapshot(
    HOLDOUT_INDEX_DATE, HOLDOUT_INDEX_DATE + pd.Timedelta(days=1), HOLDOUT_INDEX_DATE + pd.Timedelta(days=N_DAYS),
    price_bins=PRICE_BINS,
)

print(f"[Step2d] Train 스냅샷 {snap_train.shape}, Holdout 스냅샷 {snap_holdout.shape}")
print("Train treatment_source:", snap_train["treatment_source"].value_counts().to_dict())
print("Train y_repurchase_14d 양성률:", snap_train["y_repurchase_14d"].mean())
print("Holdout treatment_source:", snap_holdout["treatment_source"].value_counts().to_dict())
print("Holdout y_repurchase_14d 양성률:", snap_holdout["y_repurchase_14d"].mean())

# ---------------------------------------------------------------------------
# Step 7 (부분): 최종 저장 + 검증
# ---------------------------------------------------------------------------
OUT_COLS = [
    "회원번호", "나이", "age_band_h4", "gender_h4", "age_gender_segment", "결혼",
    "주소지_정규화", "region_key", "region_tier", "is_dawn_delivery_zone",
    "frequency", "monetary", "aov", "recency_days", "regularity_cv",
    "reward_ever_used", "reward_usage_rate",
    "preperiod_제철구매비율", "preperiod_명절선물세트구매여부",
    "가격구간", "abnormal_account_flag",
    "구독여부", "treatment_h4", "treatment_source", "구독_추정확률", "include_in_uplift_model",
    "y_repurchase_14d", "y_revenue_14d", "label_observable_flag",
]
snap_train[OUT_COLS].to_csv(BASE + "snapshot_train.csv", index=False, encoding="utf-8-sig")
snap_holdout[OUT_COLS].to_csv(BASE + "snapshot_holdout.csv", index=False, encoding="utf-8-sig")

df = df.merge(
    member_static[["회원번호", "나이", "age_band_h4", "gender_h4", "age_gender_segment"]],
    on="회원번호", how="left", suffixes=("", "_dup"),
)
ENRICHED_COLS = [
    "회원번호", "제품번호", "주문일시", "구매금액", "구매수량", "주문취소여부",
    "구독여부",  # 원본 True/False/NaN 그대로. NaN(unresolved)은 Treatment 분석에서 제외 권장
    "나이", "age_band_h4", "gender_h4", "age_gender_segment",
    "주소지_정규화", "region_key", "region_tier", "is_dawn_delivery_zone",
    "is_cancelled", "delivery_leadtime_days",
    "요일", "주말여부", "연휴전후구간", "명절선물세트여부",
    "물품중분류", "제철상품구매여부",
]
df[ENRICHED_COLS].to_csv(BASE + "merged_master_enriched.csv", index=False, encoding="utf-8-sig")

print("[Step7] 저장 완료: snapshot_train.csv, snapshot_holdout.csv, merged_master_enriched.csv")
print(f"가격구간 경계(Train 기준, AOV): {PRICE_BINS}")


[Step0] OK - shape/키/날짜범위 검증 통과


[Step1] 지역 정규화/region_key/region_tier 생성 완료
{'Tier3_배송취약권(가정)': 250652, 'Tier1_새벽배송가능권(가정)': 226124, 'Tier2_익일배송표준권(가정)': 191213, '미상': 122}


[Step1b] 명절선물세트 태깅 575건


[Step1c] {'평시': 556174, '연휴직전': 41717, '연휴직후': 39984, '연휴중': 30236}


[Step2a] 계절성 품목 54/380개 (임계값 65%, 표본<200건은 상시 처리)


[Step2b] basket_amt IQR(Train기준) = [-43696, 106012], 이상치 6.2%
스크립트 1부(Step0~2b) 완료


[Step2c] 회원 정적 속성 테이블: (12540, 12)


[Step2d] Train 스냅샷 (11732, 34), Holdout 스냅샷 (12300, 34)
Train treatment_source: {'original_false': 7895, 'unresolved': 2211, 'original_true': 1626}
Train y_repurchase_14d 양성률: 0.38825434708489603
Holdout treatment_source: {'original_false': 8276, 'unresolved': 2310, 'original_true': 1714}
Holdout y_repurchase_14d 양성률: 0.3431707317073171


[Step7] 저장 완료: snapshot_train.csv, snapshot_holdout.csv, merged_master_enriched.csv
가격구간 경계(Train 기준, AOV): [          -inf 18665.5        29962.77586207 46663.72727273
            inf]


---
## 1. run01b — L1 정규화 kitchen-sink 로지스틱 T-learner (대표 모델)

원본 컬럼 16개(고카디널리티 제외)를 전부 투입하고 `LogisticRegressionCV(L1)`로 정규화 강도를 자동
선택한다. "일부만 골라 넣기"(수동 피처선택)보다 "다 넣고 자동으로 줄이기"가 노이즈를 줄인다는 걸
확인하고 채택한 A파트 대표 모델.

In [2]:
# -*- coding: utf-8 -*-
"""
v4: '다 넣고 L1 정규화로 자동 축소' 방식의 T-learner
- 고카디널리티(region_key, 200+ 범주)만 제외하고 나머지 컬럼은 전부 투입
- 각 arm(구독자/비구독자)마다 LogisticRegressionCV(L1)로 정규화 강도(C)를 교차검증으로 자동 선택
- 결과: 0으로 수렴한 피처(=자동으로 걸러진 피처)와 살아남은 피처를 함께 리포트
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
tr = pd.read_csv(BASE + "snapshot_train.csv", encoding="utf-8-sig")
ho = pd.read_csv(BASE + "snapshot_holdout.csv", encoding="utf-8-sig")

for d in (tr, ho):
    d["log_frequency"] = np.log1p(d["frequency"])
    d["log_monetary"] = np.log1p(d["monetary"])
    d["log_aov"] = np.log1p(d["aov"])
    d["log_recency"] = np.log1p(d["recency_days"])
    d["결혼_결측표시"] = d["결혼"].fillna("결측")

# region_key(고카디널리티) 제외하고 스냅샷에 있는 사실상 모든 후보 컬럼을 투입
NUM_FEATS = [
    "log_frequency", "log_monetary", "log_aov", "log_recency",
    "regularity_cv", "reward_usage_rate", "나이",
    "preperiod_제철구매비율", "abnormal_account_flag", "reward_ever_used",
    "preperiod_명절선물세트구매여부",
]
CAT_FEATS = ["age_band_h4", "gender_h4", "region_tier", "가격구간", "결혼_결측표시"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

train_known = tr[tr["include_in_uplift_model"] == 1].copy()
X_train = pre.fit_transform(train_known[ALL_FEATS])
feat_names = pre.get_feature_names_out()
y_train = train_known["y_repurchase_14d"].values
t_train = train_known["treatment_h4"].values

print(f"[run01b] 투입 피처 수(인코딩 후): {X_train.shape[1]}개 (원본 컬럼 {len(ALL_FEATS)}개)")
print(f"[run01b] 학습표본: 구독자 {sum(t_train==1)} / 비구독자 {sum(t_train==0)}")

Cs = np.logspace(-3, 2, 15)

model_A = LogisticRegressionCV(Cs=Cs, cv=5, penalty="l1", solver="liblinear", max_iter=5000, scoring="roc_auc")
model_A.fit(X_train[t_train == 1], y_train[t_train == 1])

model_B = LogisticRegressionCV(Cs=Cs, cv=5, penalty="l1", solver="liblinear", max_iter=5000, scoring="roc_auc")
model_B.fit(X_train[t_train == 0], y_train[t_train == 0])

print(f"\n[run01b] 구독자모델(A) 선택된 C(정규화강도, 클수록 약한 규제): {model_A.C_[0]:.4f}")
print(f"[run01b] 비구독자모델(B) 선택된 C: {model_B.C_[0]:.4f}")


def report_coefs(model, name):
    coefs = model.coef_[0]
    kept = [(f, c) for f, c in zip(feat_names, coefs) if abs(c) > 1e-6]
    dropped = [f for f, c in zip(feat_names, coefs) if abs(c) <= 1e-6]
    print(f"\n[{name}] 살아남은 피처 {len(kept)}/{len(feat_names)}개 (0으로 수렴 = 자동 제외 {len(dropped)}개)")
    for f, c in sorted(kept, key=lambda x: -abs(x[1])):
        print(f"   {f}: {c:+.3f}")
    if dropped:
        print(f"   [자동 제외됨] {dropped}")


report_coefs(model_A, "구독자모델(A)")
report_coefs(model_B, "비구독자모델(B)")

# --- Holdout 평가 ---
ho_known = ho[ho["include_in_uplift_model"] == 1].copy()
X_ho_known = pre.transform(ho_known[ALL_FEATS])
sub_mask = ho_known["treatment_h4"] == 1
non_mask = ho_known["treatment_h4"] == 0
auc_A = roc_auc_score(ho_known.loc[sub_mask, "y_repurchase_14d"], model_A.predict_proba(X_ho_known[sub_mask.values])[:, 1])
auc_B = roc_auc_score(ho_known.loc[non_mask, "y_repurchase_14d"], model_B.predict_proba(X_ho_known[non_mask.values])[:, 1])
print(f"\n[run01b] Holdout AUC - 구독자모델(A): {auc_A:.3f} / 비구독자모델(B): {auc_B:.3f}")

X_ho_all = pre.transform(ho[ALL_FEATS])
uplift = model_A.predict_proba(X_ho_all)[:, 1] - model_B.predict_proba(X_ho_all)[:, 1]
ho_tmp = ho.copy()
ho_tmp["uplift"] = uplift
target = ho_tmp[ho_tmp["treatment_h4"] == 0]

rng = np.random.default_rng(42)
vals = target["uplift"].values
boot = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"\n[run01b] 비구독자 전체 평균 Uplift: {vals.mean()*100:.3f}%p, 95%CI=[{lo*100:.2f}, {hi*100:.2f}]%p")
print(f"[run01b] Uplift 표준편차: {vals.std()*100:.3f}%p, 최댓값: {vals.max()*100:.2f}%p")

seg = target.groupby("가격구간", observed=True)["uplift"].mean() * 100
print("\n[run01b] 가격구간별 평균 Uplift(%p):")
print(seg)

ho_tmp[["회원번호", "가격구간", "treatment_h4", "uplift"]].to_csv(
    BASE + "run01b_uplift_scores_holdout.csv", index=False, encoding="utf-8-sig"
)
print("\n저장 완료: run01b_uplift_scores_holdout.csv")


[run01b] 투입 피처 수(인코딩 후): 29개 (원본 컬럼 16개)
[run01b] 학습표본: 구독자 1626 / 비구독자 7895


C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line


[run01b] 구독자모델(A) 선택된 C(정규화강도, 클수록 약한 규제): 0.0611
[run01b] 비구독자모델(B) 선택된 C: 0.0118

[구독자모델(A)] 살아남은 피처 4/29개 (0으로 수렴 = 자동 제외 25개)
   num__log_frequency: +1.388
   num__log_recency: -0.491
   cat__gender_h4_F: -0.420
   num__reward_ever_used: +0.077
   [자동 제외됨] ['num__log_monetary', 'num__log_aov', 'num__regularity_cv', 'num__reward_usage_rate', 'num__나이', 'num__preperiod_제철구매비율', 'num__abnormal_account_flag', 'num__preperiod_명절선물세트구매여부', 'cat__age_band_h4_30s', 'cat__age_band_h4_40s', 'cat__age_band_h4_50s', 'cat__age_band_h4_60+', 'cat__age_band_h4_<30', 'cat__gender_h4_M', 'cat__gender_h4_Unknown', 'cat__region_tier_Tier1_새벽배송가능권(가정)', 'cat__region_tier_Tier2_익일배송표준권(가정)', 'cat__region_tier_Tier3_배송취약권(가정)', 'cat__가격구간_Q1_저가', 'cat__가격구간_Q2', 'cat__가격구간_Q3', 'cat__가격구간_Q4_고가', 'cat__결혼_결측표시_결측', 'cat__결혼_결측표시_기혼', 'cat__결혼_결측표시_미혼']

[비구독자모델(B)] 살아남은 피처 3/29개 (0으로 수렴 = 자동 제외 26개)
   num__log_frequency: +1.338
   num__log_recency: -0.366
   num__log_monetary: +0.047
   [자동 제외됨] ['num


[run01b] 비구독자 전체 평균 Uplift: 0.677%p, 95%CI=[0.62, 0.73]%p
[run01b] Uplift 표준편차: 2.667%p, 최댓값: 16.05%p

[run01b] 가격구간별 평균 Uplift(%p):
가격구간
Q1_저가    0.273408
Q2       0.812959
Q3       0.881830
Q4_고가    0.739170
Name: uplift, dtype: float64

저장 완료: run01b_uplift_scores_holdout.csv


---
## 2. run08b — RandomForest T-learner (GridSearchCV 재튜닝)

run01b와 같은 kitchen-sink 피처셋을 RandomForest로 학습. 로지스틱과 가격구간 1위가 다르게 나오는데
(로지스틱=Q3, RF=Q4), 이게 하이퍼파라미터 미튜닝 때문이 아니라 **모델 클래스 자체의 차이**임을
GridSearchCV 재튜닝 전후 비교로 확인했다.

In [3]:
# -*- coding: utf-8 -*-
"""
run08b: Uplift Forest 재튜닝판 — run08(수동 하이퍼파라미터 고정)을 run06/run07과 동일하게
GridSearchCV로 정식 튜닝. run01b(로지스틱)를 "대표모델"로 정할 때 비교 대상이던 run08이
사실 튜닝이 안 된 상태였다는 걸 8/24 뒤늦게 인지해서 재실행(사용자 확인, 백테스팅 착수 전).
- 피처셋은 run06/run07과 동일(log_aov 제외 — log_monetary=log_frequency+log_aov 완전종속 문제, 8/24 발견)
- TwoModels 래퍼 대신 run07과 같은 방식으로 구독자/비구독자 RF를 직접 GridSearchCV로 튜닝
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, export_text

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
tr = pd.read_csv(BASE + "snapshot_train.csv", encoding="utf-8-sig")
ho = pd.read_csv(BASE + "snapshot_holdout.csv", encoding="utf-8-sig")

for d in (tr, ho):
    d["log_frequency"] = np.log1p(d["frequency"])
    d["log_monetary"] = np.log1p(d["monetary"])
    d["log_recency"] = np.log1p(d["recency_days"])  # log_aov는 완전종속 문제로 제외(run06과 동일)
    d["결혼_결측표시"] = d["결혼"].fillna("결측")

NUM_FEATS = [
    "log_frequency", "log_monetary", "log_recency",
    "regularity_cv", "reward_usage_rate", "나이",
    "preperiod_제철구매비율", "abnormal_account_flag", "reward_ever_used",
    "preperiod_명절선물세트구매여부",
]
CAT_FEATS = ["age_band_h4", "gender_h4", "region_tier", "가격구간", "결혼_결측표시"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

train_known = tr[tr["include_in_uplift_model"] == 1].copy()
X_train = pre.fit_transform(train_known[ALL_FEATS])
feat_names = pre.get_feature_names_out()
y_train = train_known["y_repurchase_14d"].values
t_train = train_known["treatment_h4"].values

print(f"[run08b] 학습표본: 구독자 {sum(t_train==1)} / 비구독자 {sum(t_train==0)}")

# ================= RandomForest + GridSearchCV (run06/run07과 동일 방법론) =================
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [10, 30, 50, 100],
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Windows에서 GridSearchCV+RandomForest 동시 n_jobs=-1 중첩병렬화 시 TerminatedWorkerError 발생 -> RF는 n_jobs=1
gs_A = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="roc_auc", n_jobs=-1)
gs_A.fit(X_train[t_train == 1], y_train[t_train == 1])
gs_B = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="roc_auc", n_jobs=-1)
gs_B.fit(X_train[t_train == 0], y_train[t_train == 0])
print(f"[GridSearchCV] 구독자모델 최적: {gs_A.best_params_} (CV AUC={gs_A.best_score_:.3f})")
print(f"[GridSearchCV] 비구독자모델 최적: {gs_B.best_params_} (CV AUC={gs_B.best_score_:.3f})")
rf_A, rf_B = gs_A.best_estimator_, gs_B.best_estimator_

# ================= Holdout 적용 + Uplift 비교 =================
X_ho_all = pre.transform(ho[ALL_FEATS])
uplift_forest = rf_A.predict_proba(X_ho_all)[:, 1] - rf_B.predict_proba(X_ho_all)[:, 1]
ho = ho.copy()
ho["uplift_forest"] = uplift_forest

target = ho[ho["treatment_h4"] == 0]
vals = target["uplift_forest"].values
rng = np.random.default_rng(42)
boot = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])

print("\n=== run08b: Uplift Forest (RandomForest, GridSearchCV 튜닝) ===")
print(f"평균 Uplift: {vals.mean()*100:.3f}%p, 95%CI=[{lo*100:.2f}, {hi*100:.2f}]%p")
print(f"표준편차: {vals.std()*100:.3f}%p, 최댓값: {vals.max()*100:.2f}%p, 최솟값: {vals.min()*100:.2f}%p")

seg = target.groupby("가격구간", observed=True)["uplift_forest"].mean() * 100
print("\n가격구간별 평균 Uplift(%p):")
print(seg)

# --- RandomForest 피처 중요도 (두 arm 평균) ---
imp_t = pd.Series(rf_A.feature_importances_, index=feat_names)
imp_c = pd.Series(rf_B.feature_importances_, index=feat_names)
imp_avg = ((imp_t + imp_c) / 2).sort_values(ascending=False)
print("\n피처 중요도 상위 10 (구독자/비구독자 모델 평균):")
print(imp_avg.head(10))

# --- 해석용 서로게이트 트리 ---
X_ho_df = pd.DataFrame(X_ho_all.toarray() if hasattr(X_ho_all, "toarray") else X_ho_all, columns=feat_names)
surrogate = DecisionTreeRegressor(max_depth=3, min_samples_leaf=300, random_state=42)
surrogate.fit(X_ho_df, ho["uplift_forest"])
r2 = surrogate.score(X_ho_df, ho["uplift_forest"])
print(f"\n=== 해석용 서로게이트 트리 (Uplift 점수 근사, R^2={r2:.3f}) ===")
print(export_text(surrogate, feature_names=list(feat_names), max_depth=3))

ho[["회원번호", "가격구간", "treatment_h4", "uplift_forest"]].to_csv(
    BASE + "run08b_uplift_forest_scores.csv", index=False, encoding="utf-8-sig"
)
print("\n저장 완료: run08b_uplift_forest_scores.csv")


[run08b] 학습표본: 구독자 1626 / 비구독자 7895


[GridSearchCV] 구독자모델 최적: {'max_depth': 5, 'min_samples_leaf': 30, 'n_estimators': 400} (CV AUC=0.883)
[GridSearchCV] 비구독자모델 최적: {'max_depth': 7, 'min_samples_leaf': 10, 'n_estimators': 200} (CV AUC=0.869)



=== run08b: Uplift Forest (RandomForest, GridSearchCV 튜닝) ===
평균 Uplift: 1.547%p, 95%CI=[1.45, 1.64]%p
표준편차: 4.414%p, 최댓값: 22.76%p, 최솟값: -19.25%p

가격구간별 평균 Uplift(%p):
가격구간
Q1_저가    1.029564
Q2       0.759386
Q3       1.641630
Q4_고가    2.731969
Name: uplift_forest, dtype: float64

피처 중요도 상위 10 (구독자/비구독자 모델 평균):
num__log_frequency        0.326188
num__log_monetary         0.207348
num__log_recency          0.177844
num__preperiod_제철구매비율     0.082997
num__reward_usage_rate    0.074671
num__reward_ever_used     0.058453
num__regularity_cv        0.033774
num__나이                   0.009355
cat__가격구간_Q1_저가           0.005216
cat__가격구간_Q4_고가           0.003269
dtype: float64



=== 해석용 서로게이트 트리 (Uplift 점수 근사, R^2=0.280) ===
|--- num__log_frequency <= -0.69
|   |--- num__log_frequency <= -1.00
|   |   |--- num__log_recency <= 0.78
|   |   |   |--- value: [0.04]
|   |   |--- num__log_recency >  0.78
|   |   |   |--- value: [0.03]
|   |--- num__log_frequency >  -1.00
|   |   |--- cat__region_tier_Tier3_배송취약권(가정) <= 0.50
|   |   |   |--- value: [0.03]
|   |   |--- cat__region_tier_Tier3_배송취약권(가정) >  0.50
|   |   |   |--- value: [0.01]
|--- num__log_frequency >  -0.69
|   |--- num__reward_usage_rate <= -0.43
|   |   |--- num__log_frequency <= 0.11
|   |   |   |--- value: [-0.00]
|   |   |--- num__log_frequency >  0.11
|   |   |   |--- value: [-0.05]
|   |--- num__reward_usage_rate >  -0.43
|   |   |--- num__preperiod_제철구매비율 <= -0.44
|   |   |   |--- value: [-0.01]
|   |   |--- num__preperiod_제철구매비율 >  -0.44
|   |   |   |--- value: [0.02]


저장 완료: run08b_uplift_forest_scores.csv


---
## 3. run06 — 매출 관점 Uplift (Y = 14일 내 재구매 금액)

재구매 "확률"이 아니라 "금액" 기준으로 Uplift를 다시 계산한다. `log_frequency·log_monetary·log_aov`가
`monetary=frequency×aov` 항등식으로 완전 종속(perfect collinearity)이던 버그를 발견해 `log_aov`를
제거하고 재실행했다.

In [4]:
# -*- coding: utf-8 -*-
"""
run06: 매출 관점 Uplift — Y=y_revenue_14d(14일 내 재구매 금액, 원)
- kitchen-sink 피처(run01b/run07과 동일 철학) + 이번엔 RandomForest도 정식 그리드서치로 튜닝
- 선형: ElasticNetCV(alpha, l1_ratio 교차검증) / 트리: RandomForestRegressor + GridSearchCV
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
tr = pd.read_csv(BASE + "snapshot_train.csv", encoding="utf-8-sig")
ho = pd.read_csv(BASE + "snapshot_holdout.csv", encoding="utf-8-sig")

for d in (tr, ho):
    d["log_frequency"] = np.log1p(d["frequency"])
    d["log_monetary"] = np.log1p(d["monetary"])
    d["log_aov"] = np.log1p(d["aov"])
    d["log_recency"] = np.log1p(d["recency_days"])
    d["결혼_결측표시"] = d["결혼"].fillna("결측")

NUM_FEATS = [
    # log_aov 제외: monetary = frequency * aov 이므로 log_monetary = log_frequency + log_aov (완전 종속, 8/24 발견)
    "log_frequency", "log_monetary", "log_recency",
    "regularity_cv", "reward_usage_rate", "나이",
    "preperiod_제철구매비율", "abnormal_account_flag", "reward_ever_used",
    "preperiod_명절선물세트구매여부",
]
CAT_FEATS = ["age_band_h4", "gender_h4", "region_tier", "가격구간", "결혼_결측표시"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

train_known = tr[tr["include_in_uplift_model"] == 1].copy()
X_train = pre.fit_transform(train_known[ALL_FEATS])
feat_names = pre.get_feature_names_out()
y_train = train_known["y_revenue_14d"].values
t_train = train_known["treatment_h4"].values

print(f"[run06] 학습표본: 구독자 {sum(t_train==1)} / 비구독자 {sum(t_train==0)}")
print(f"[run06] Y(매출) 분포 - 평균 {y_train.mean():.0f}원, 중앙값 {np.median(y_train):.0f}원, 0인 비율 {(y_train==0).mean()*100:.1f}%")

# ================= 선형: ElasticNetCV =================
enet_A = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, 1.0], alphas=np.logspace(-1, 4, 20), cv=5, max_iter=20000)
enet_A.fit(X_train[t_train == 1], y_train[t_train == 1])
enet_B = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, 1.0], alphas=np.logspace(-1, 4, 20), cv=5, max_iter=20000)
enet_B.fit(X_train[t_train == 0], y_train[t_train == 0])

print(f"\n[ElasticNetCV] 구독자모델 alpha={enet_A.alpha_:.2f}, l1_ratio={enet_A.l1_ratio_:.2f}")
print(f"[ElasticNetCV] 비구독자모델 alpha={enet_B.alpha_:.2f}, l1_ratio={enet_B.l1_ratio_:.2f}")
kept_A = [(f, c) for f, c in zip(feat_names, enet_A.coef_) if abs(c) > 1e-6]
kept_B = [(f, c) for f, c in zip(feat_names, enet_B.coef_) if abs(c) > 1e-6]
print(f"[ElasticNetCV] 구독자모델 생존피처 {len(kept_A)}/{len(feat_names)}: {sorted(kept_A, key=lambda x:-abs(x[1]))[:8]}")
print(f"[ElasticNetCV] 비구독자모델 생존피처 {len(kept_B)}/{len(feat_names)}: {sorted(kept_B, key=lambda x:-abs(x[1]))[:8]}")

# ================= 트리: RandomForestRegressor + GridSearchCV =================
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [10, 30, 50, 100],
}
cv5 = KFold(n_splits=5, shuffle=True, random_state=42)

# Windows에서 GridSearchCV와 RandomForest를 동시에 n_jobs=-1로 중첩 병렬화하면
# joblib 워커가 죽는 문제(TerminatedWorkerError) 발생 -> 바깥(GridSearchCV)만 병렬화
gs_A = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="neg_mean_squared_error", n_jobs=-1)
gs_A.fit(X_train[t_train == 1], y_train[t_train == 1])
gs_B = GridSearchCV(RandomForestRegressor(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="neg_mean_squared_error", n_jobs=-1)
gs_B.fit(X_train[t_train == 0], y_train[t_train == 0])

print(f"\n[GridSearchCV] 구독자모델 최적 파라미터: {gs_A.best_params_} (CV RMSE={np.sqrt(-gs_A.best_score_):.0f}원)")
print(f"[GridSearchCV] 비구독자모델 최적 파라미터: {gs_B.best_params_} (CV RMSE={np.sqrt(-gs_B.best_score_):.0f}원)")

rf_A, rf_B = gs_A.best_estimator_, gs_B.best_estimator_

# ================= Holdout 적용 + Uplift 비교 =================
X_ho_all = pre.transform(ho[ALL_FEATS])

uplift_enet = enet_A.predict(X_ho_all) - enet_B.predict(X_ho_all)
uplift_rf = rf_A.predict(X_ho_all) - rf_B.predict(X_ho_all)

ho = ho.copy()
ho["uplift_revenue_enet"] = uplift_enet
ho["uplift_revenue_rf"] = uplift_rf

target = ho[ho["treatment_h4"] == 0]
rng = np.random.default_rng(42)


def boot_ci(vals, n=2000):
    b = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])


for col, label in [("uplift_revenue_enet", "ElasticNet(선형)"), ("uplift_revenue_rf", "RandomForest(그리드서치)")]:
    vals = target[col].values
    lo, hi = boot_ci(vals)
    print(f"\n=== {label} 매출 Uplift ===")
    print(f"평균: {vals.mean():.0f}원/14일, 95%CI=[{lo:.0f}, {hi:.0f}]원, 0포함여부: {'포함(비유의)' if lo<0<hi else '0 미포함(유의)'}")
    seg = target.groupby("가격구간", observed=True)[col].mean()
    print("가격구간별 평균 매출Uplift(원):")
    print(seg)

ho[["회원번호", "가격구간", "treatment_h4", "uplift_revenue_enet", "uplift_revenue_rf"]].to_csv(
    BASE + "run06_revenue_uplift_scores.csv", index=False, encoding="utf-8-sig"
)
print("\n저장 완료: run06_revenue_uplift_scores.csv")


[run06] 학습표본: 구독자 1626 / 비구독자 7895
[run06] Y(매출) 분포 - 평균 34415원, 중앙값 0원, 0인 비율 61.3%



[ElasticNetCV] 구독자모델 alpha=483.29, l1_ratio=1.00
[ElasticNetCV] 비구독자모델 alpha=263.67, l1_ratio=1.00
[ElasticNetCV] 구독자모델 생존피처 18/28: [('num__log_frequency', np.float64(34212.27068378354)), ('cat__가격구간_Q4_고가', np.float64(24627.065638496166)), ('num__abnormal_account_flag', np.float64(15901.687398334123)), ('cat__가격구간_Q2', np.float64(-8365.040421354593)), ('num__log_recency', np.float64(-5675.994889333164)), ('num__reward_ever_used', np.float64(-4578.788190631763)), ('cat__가격구간_Q1_저가', np.float64(-4445.07502979187)), ('num__preperiod_명절선물세트구매여부', np.float64(4424.998339495806))]
[ElasticNetCV] 비구독자모델 생존피처 15/28: [('num__log_frequency', np.float64(39284.22493116989)), ('cat__가격구간_Q4_고가', np.float64(30717.16427641481)), ('num__abnormal_account_flag', np.float64(22394.103205894084)), ('num__reward_ever_used', np.float64(-9927.66949780243)), ('num__preperiod_명절선물세트구매여부', np.float64(7657.639001190067)), ('num__reward_usage_rate', np.float64(6588.176912105258)), ('cat__가격구간_Q2', np.float64(-580


[GridSearchCV] 구독자모델 최적 파라미터: {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 200} (CV RMSE=55419원)
[GridSearchCV] 비구독자모델 최적 파라미터: {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 200} (CV RMSE=60379원)



=== ElasticNet(선형) 매출 Uplift ===
평균: -934원/14일, 95%CI=[-1155, -720]원, 0포함여부: 0 미포함(유의)
가격구간별 평균 매출Uplift(원):
가격구간
Q1_저가     456.181907
Q2       -122.195314
Q3       1406.063840
Q4_고가   -5359.771583
Name: uplift_revenue_enet, dtype: float64



=== RandomForest(그리드서치) 매출 Uplift ===
평균: -621원/14일, 95%CI=[-1240, -2]원, 0포함여부: 0 미포함(유의)
가격구간별 평균 매출Uplift(원):
가격구간
Q1_저가    1044.864493
Q2       1222.990645
Q3       1106.097641
Q4_고가   -5732.333553
Name: uplift_revenue_rf, dtype: float64

저장 완료: run06_revenue_uplift_scores.csv


---
## 4. run07 — 이탈위험군별 재구매 Uplift

고객을 Active(&lt;30일)/관심필요(30~89일)/이탈위험(90일+)로 나눠 "이탈위험군일수록 구독의 재구매
효과가 큰가"를 검증한다. 로지스틱과 RandomForest가 이탈위험(90일+) 구간에서 방향이 정반대로 나와
**결론을 유보**했다 — 원자료 재구매율 차이가 사실상 0(3.231% vs 3.232%)이라 두 모델이 미미한 신호를
서로 다른 방향으로 과대추정한 것으로 판단.

In [5]:
# -*- coding: utf-8 -*-
"""
run07: 이탈 방지 관점 보조 타겟 (Y=y_repurchase_14d, X를 recency_days 기준 위험군으로 슬라이스)
- 목표 슬라이드 기준 그대로: Active(<30일) / 관심필요(30~89일) / 이탈위험(90일+)
- "이탈 위험군일수록 구독의 재구매 Uplift가 더 큰가"를 검증 -> 맞으면 "복귀 프로모션으로 구독을 미는" 전략 근거
- kitchen-sink 피처 + LogisticRegressionCV(L1) + RandomForestClassifier(GridSearchCV, 정식 튜닝) 둘 다 사용
"""
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
tr = pd.read_csv(BASE + "snapshot_train.csv", encoding="utf-8-sig")
ho = pd.read_csv(BASE + "snapshot_holdout.csv", encoding="utf-8-sig")

for d in (tr, ho):
    d["log_frequency"] = np.log1p(d["frequency"])
    d["log_monetary"] = np.log1p(d["monetary"])
    d["log_recency"] = np.log1p(d["recency_days"])  # log_aov는 8/24 발견한 완전종속 문제로 제외
    d["결혼_결측표시"] = d["결혼"].fillna("결측")

    def risk_band(days):
        if days < 30:
            return "Active(<30일)"
        if days < 90:
            return "관심필요(30~89일)"
        return "이탈위험(90일+)"

    d["이탈위험군"] = d["recency_days"].apply(risk_band)

NUM_FEATS = [
    "log_frequency", "log_monetary", "log_recency",
    "regularity_cv", "reward_usage_rate", "나이",
    "preperiod_제철구매비율", "abnormal_account_flag", "reward_ever_used",
    "preperiod_명절선물세트구매여부",
]
CAT_FEATS = ["age_band_h4", "gender_h4", "region_tier", "가격구간", "결혼_결측표시"]
ALL_FEATS = NUM_FEATS + CAT_FEATS

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_FEATS),
])

train_known = tr[tr["include_in_uplift_model"] == 1].copy()
X_train = pre.fit_transform(train_known[ALL_FEATS])
feat_names = pre.get_feature_names_out()
y_train = train_known["y_repurchase_14d"].values
t_train = train_known["treatment_h4"].values

print(f"[run07] 학습표본: 구독자 {sum(t_train==1)} / 비구독자 {sum(t_train==0)}")
print("[run07] Train 이탈위험군 분포:")
print(train_known["이탈위험군"].value_counts())

# ================= 로지스틱 L1 (kitchen-sink, run01b와 동일 방법론) =================
Cs = np.logspace(-3, 2, 15)
logit_A = LogisticRegressionCV(Cs=Cs, cv=5, penalty="l1", solver="liblinear", max_iter=5000, scoring="roc_auc")
logit_A.fit(X_train[t_train == 1], y_train[t_train == 1])
logit_B = LogisticRegressionCV(Cs=Cs, cv=5, penalty="l1", solver="liblinear", max_iter=5000, scoring="roc_auc")
logit_B.fit(X_train[t_train == 0], y_train[t_train == 0])
print(f"\n[LogisticRegressionCV] 구독자모델 C={logit_A.C_[0]:.4f}, 비구독자모델 C={logit_B.C_[0]:.4f}")

# ================= RandomForest + GridSearchCV (run06과 동일 방법론) =================
param_grid = {
    "n_estimators": [200, 400],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [10, 30, 50, 100],
}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

gs_A = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="roc_auc", n_jobs=-1)
gs_A.fit(X_train[t_train == 1], y_train[t_train == 1])
gs_B = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=1), param_grid, cv=cv5, scoring="roc_auc", n_jobs=-1)
gs_B.fit(X_train[t_train == 0], y_train[t_train == 0])
print(f"[GridSearchCV] 구독자모델 최적: {gs_A.best_params_} (CV AUC={gs_A.best_score_:.3f})")
print(f"[GridSearchCV] 비구독자모델 최적: {gs_B.best_params_} (CV AUC={gs_B.best_score_:.3f})")
rf_A, rf_B = gs_A.best_estimator_, gs_B.best_estimator_

# ================= Holdout 평가 =================
ho_known = ho[ho["include_in_uplift_model"] == 1].copy()
X_ho_known = pre.transform(ho_known[ALL_FEATS])
sub_mask = ho_known["treatment_h4"] == 1
non_mask = ho_known["treatment_h4"] == 0

for name, mA, mB in [("로지스틱L1", logit_A, logit_B), ("RandomForest", rf_A, rf_B)]:
    auc_A = roc_auc_score(ho_known.loc[sub_mask, "y_repurchase_14d"], mA.predict_proba(X_ho_known[sub_mask.values])[:, 1])
    auc_B = roc_auc_score(ho_known.loc[non_mask, "y_repurchase_14d"], mB.predict_proba(X_ho_known[non_mask.values])[:, 1])
    print(f"[{name}] Holdout AUC - 구독자모델: {auc_A:.3f} / 비구독자모델: {auc_B:.3f}")

X_ho_all = pre.transform(ho[ALL_FEATS])
ho = ho.copy()
ho["uplift_logit"] = logit_A.predict_proba(X_ho_all)[:, 1] - logit_B.predict_proba(X_ho_all)[:, 1]
ho["uplift_rf"] = rf_A.predict_proba(X_ho_all)[:, 1] - rf_B.predict_proba(X_ho_all)[:, 1]

target = ho[ho["treatment_h4"] == 0]  # 비구독자(실제 복귀프로모션 타겟 후보)
rng = np.random.default_rng(42)


def boot_ci(vals, n=2000):
    b = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])


order = ["Active(<30일)", "관심필요(30~89일)", "이탈위험(90일+)"]
print("\n=== 이탈위험군별 재구매 Uplift (H7: 이탈위험군일수록 Uplift가 큰가) ===")
for col, label in [("uplift_logit", "로지스틱L1"), ("uplift_rf", "RandomForest")]:
    print(f"\n--- {label} ---")
    for seg in order:
        vals = target.loc[target["이탈위험군"] == seg, col].values
        lo, hi = boot_ci(vals)
        sig = "0 미포함(유의)" if lo > 0 or hi < 0 else "포함(비유의)"
        print(f"  {seg}: n={len(vals)}, 평균Uplift={vals.mean()*100:.2f}%p, 95%CI=[{lo*100:.2f}, {hi*100:.2f}]%p, {sig}")

ho[["회원번호", "이탈위험군", "가격구간", "treatment_h4", "uplift_logit", "uplift_rf"]].to_csv(
    BASE + "run07_churn_uplift_scores.csv", index=False, encoding="utf-8-sig"
)
print("\n저장 완료: run07_churn_uplift_scores.csv")


[run07] 학습표본: 구독자 1626 / 비구독자 7895
[run07] Train 이탈위험군 분포:
이탈위험군
Active(<30일)    4973
이탈위험(90일+)      2499
관심필요(30~89일)    2049
Name: count, dtype: int64


C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line

C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2092: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:2123: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratios' and 'Cs' instead. Use l1_ratios=(0,) instead of penalty='l2', l1_ratios=(1,) instead of penalty='l1', l1_ratios set to floats between 0 and 1 instead of penalty='elasticnet', and Cs=(np.inf,) instead of penalty=None.
  warnings.warn(
C:\Users\aidan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\line


[LogisticRegressionCV] 구독자모델 C=0.0611, 비구독자모델 C=0.0118


[GridSearchCV] 구독자모델 최적: {'max_depth': 5, 'min_samples_leaf': 30, 'n_estimators': 400} (CV AUC=0.883)
[GridSearchCV] 비구독자모델 최적: {'max_depth': 7, 'min_samples_leaf': 10, 'n_estimators': 200} (CV AUC=0.869)
[로지스틱L1] Holdout AUC - 구독자모델: 0.881 / 비구독자모델: 0.883


...


=== 이탈위험군별 재구매 Uplift (H7: 이탈위험군일수록 Uplift가 큰가) ===

--- 로지스틱L1 ---


  Active(<30일): n=4121, 평균Uplift=2.32%p, 95%CI=[2.25, 2.40]%p, 0 미포함(유의)
  관심필요(30~89일): n=1703, 평균Uplift=-0.48%p, 95%CI=[-0.59, -0.38]%p, 0 미포함(유의)


  이탈위험(90일+): n=2452, 평균Uplift=-1.31%p, 95%CI=[-1.36, -1.26]%p, 0 미포함(유의)

--- RandomForest ---


  Active(<30일): n=4121, 평균Uplift=0.88%p, 95%CI=[0.73, 1.03]%p, 0 미포함(유의)
  관심필요(30~89일): n=1703, 평균Uplift=1.81%p, 95%CI=[1.57, 2.04]%p, 0 미포함(유의)


  이탈위험(90일+): n=2452, 평균Uplift=2.49%p, 95%CI=[2.38, 2.61]%p, 0 미포함(유의)

저장 완료: run07_churn_uplift_scores.csv


---
## 5. run09 — 백테스팅 (A+B+C+D 4파트 통합)

A(run01b/run08b)와 B(H4 연령성별)·C(H3 계절명절)·D(H2 지역배송)가 각자 다른 가설·피처로 만든 개인별
Uplift 점수 8개를 회원번호 기준으로 병합해, "같은 고객을 Persuadable로 보는가"를 Spearman 순위상관과
상위 20% Jaccard 겹침으로 검증한다.

In [6]:
# -*- coding: utf-8 -*-
"""
run09: A/B/C/D 개인별 Uplift 점수 백테스팅
- "각 파트 모델이 같은 고객을 Persuadable(타겟 우선순위 높음)로 보는가"를 회원번호 기준 병합해 확인
- 비교 대상 8개 스코어: A(run01b 로지스틱 대표모델 / run08b RF 재튜닝판), B/C/D(각 logit/rf)
- 대상 모집단: treatment_h4==0(비구독자, 실제 타겟팅 후보군) — 각 파트 스크립트가 전부 이 기준으로 Uplift를 계산했음
- 두 지표: (1) Spearman 순위상관 — 전반적 순서가 비슷한가 (2) 상위 20% Jaccard 겹침 — 실제 타겟 후보군이 겹치는가
"""
import numpy as np
import pandas as pd

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
TOP_PCT = 0.20

a_logit = pd.read_csv(BASE + "run01b_uplift_scores_holdout.csv", encoding="utf-8-sig")
a_rf = pd.read_csv(BASE + "run08b_uplift_forest_scores.csv", encoding="utf-8-sig")
b_logit = pd.read_csv(BASE + "uplift_scores_partB_logit.csv", encoding="utf-8-sig")
b_rf = pd.read_csv(BASE + "uplift_scores_partB_rf.csv", encoding="utf-8-sig")
c = pd.read_csv(BASE + "uplift_scores_partC.csv", encoding="utf-8-sig")
d = pd.read_csv(BASE + "uplift_scores_partD.csv", encoding="utf-8-sig")

# --- 회원번호 기준 병합 준비: 각 파일에서 uplift 컬럼만 추출해 파트_모델명으로 통일 ---
frames = {
    "A_logit": a_logit[["회원번호", "treatment_h4", "uplift"]].rename(columns={"uplift": "uplift_A_logit"}),
    "A_rf": a_rf[["회원번호", "treatment_h4", "uplift_forest"]].rename(columns={"uplift_forest": "uplift_A_rf"}),
    "B_logit": b_logit[["회원번호", "treatment_h4", "uplift"]].rename(columns={"uplift": "uplift_B_logit"}),
    "B_rf": b_rf[["회원번호", "treatment_h4", "uplift"]].rename(columns={"uplift": "uplift_B_rf"}),
    "C_logit": c[["회원번호", "treatment_h4", "uplift_logit"]].rename(columns={"uplift_logit": "uplift_C_logit"}),
    "C_rf": c[["회원번호", "treatment_h4", "uplift_rf"]].rename(columns={"uplift_rf": "uplift_C_rf"}),
    "D_logit": d[["회원번호", "treatment_h4", "uplift_logit"]].rename(columns={"uplift_logit": "uplift_D_logit"}),
    "D_rf": d[["회원번호", "treatment_h4", "uplift_rf"]].rename(columns={"uplift_rf": "uplift_D_rf"}),
}

# --- treatment_h4 정합성 체크: 회원번호가 같으면 4개 파트에서 treatment_h4도 같아야 함 ---
# 주의: treatment_h4는 구독여부 unresolved(18.7%)인 사람은 NaN이고, pandas에서 NaN != NaN은
# 항상 True로 평가되므로 단순 != 비교는 오탐을 냄. NaN을 결측 그대로 비교(둘 다 NaN이면 일치로 간주)해야 함.
merged = frames["A_logit"][["회원번호", "treatment_h4"]].rename(columns={"treatment_h4": "treatment_h4_A"})
for key in ["B_logit", "C_logit", "D_logit"]:
    part = key.split("_")[0]
    merged = merged.merge(
        frames[key][["회원번호", "treatment_h4"]].rename(columns={"treatment_h4": f"treatment_h4_{part}"}),
        on="회원번호", how="inner",
    )
cols4 = ["treatment_h4_A", "treatment_h4_B", "treatment_h4_C", "treatment_h4_D"]
agree = merged[cols4].apply(lambda row: row.dropna().nunique() <= 1, axis=1)
real_conflict = merged[~agree]
na_pattern_consistent = merged[cols4].isna().nunique(axis=1).eq(1).all()
print(f"[정합성 체크] inner join 대상 {len(merged)}명 중 진짜 값 충돌(0 vs 1): {len(real_conflict)}명 "
      f"(NaN 패턴은 4개 파트 전부 동일: {na_pattern_consistent})")

# --- 8개 uplift 점수를 회원번호 기준 inner join으로 병합 ---
score = frames["A_logit"][["회원번호", "treatment_h4", "uplift_A_logit"]]
for key in ["A_rf", "B_logit", "B_rf", "C_logit", "C_rf", "D_logit", "D_rf"]:
    col = f"uplift_{key}"
    score = score.merge(frames[key][["회원번호", col]], on="회원번호", how="inner")

print(f"[병합] inner join 최종 표본: {len(score)}명 (A/B/C=12,391명, D=12,388명 중 교집합)")

score.to_csv(BASE + "run09_backtesting_merged_scores.csv", index=False, encoding="utf-8-sig")

# --- 타겟 모집단: 비구독자(treatment_h4==0)만 대상 (각 파트 스크립트의 target 정의와 동일) ---
target = score[score["treatment_h4"] == 0].copy()
score_cols = ["uplift_A_logit", "uplift_A_rf", "uplift_B_logit", "uplift_B_rf",
              "uplift_C_logit", "uplift_C_rf", "uplift_D_logit", "uplift_D_rf"]
print(f"[타겟 모집단] 비구독자 {len(target)}명 기준으로 상관/겹침 분석")

# ================= 1) Spearman 순위상관 =================
corr = target[score_cols].corr(method="spearman")
print("\n=== 1) Spearman 순위상관 행렬 ===")
print(corr.round(3))
corr.to_csv(BASE + "run09_spearman_correlation.csv", encoding="utf-8-sig")

# ================= 2) 상위 20% Jaccard 겹침 =================
n_top = int(len(target) * TOP_PCT)
top_sets = {col: set(target.nlargest(n_top, col)["회원번호"]) for col in score_cols}

jac = pd.DataFrame(index=score_cols, columns=score_cols, dtype=float)
for r in score_cols:
    for c_ in score_cols:
        inter = len(top_sets[r] & top_sets[c_])
        union = len(top_sets[r] | top_sets[c_])
        jac.loc[r, c_] = inter / union if union else np.nan

print(f"\n=== 2) 상위 {int(TOP_PCT*100)}%(n={n_top}) Jaccard 겹침 행렬 ===")
print(jac.round(3))
jac.to_csv(BASE + "run09_top20pct_jaccard.csv", encoding="utf-8-sig")

# --- 4개 파트(A/B/C/D) 모두의 상위20%에 동시에 들어가는 "만장일치 Persuadable" 고객 수 ---
all_four_logit = top_sets["uplift_A_logit"] & top_sets["uplift_B_logit"] & top_sets["uplift_C_logit"] & top_sets["uplift_D_logit"]
all_four_rf = top_sets["uplift_A_rf"] & top_sets["uplift_B_rf"] & top_sets["uplift_C_rf"] & top_sets["uplift_D_rf"]
expected_if_random = len(target) * (TOP_PCT ** 4)
print(f"\n[만장일치] logit 4개 파트 상위20% 교집합: {len(all_four_logit)}명 (무작위 기대치 {expected_if_random:.1f}명)")
print(f"[만장일치] rf 4개 파트 상위20% 교집합: {len(all_four_rf)}명 (무작위 기대치 {expected_if_random:.1f}명)")

print("\n저장 완료: run09_backtesting_merged_scores.csv, run09_spearman_correlation.csv, run09_top20pct_jaccard.csv")


[정합성 체크] inner join 대상 12300명 중 진짜 값 충돌(0 vs 1): 0명 (NaN 패턴은 4개 파트 전부 동일: True)
[병합] inner join 최종 표본: 12300명 (A/B/C=12,391명, D=12,388명 중 교집합)


[타겟 모집단] 비구독자 8276명 기준으로 상관/겹침 분석

=== 1) Spearman 순위상관 행렬 ===
                uplift_A_logit  uplift_A_rf  uplift_B_logit  uplift_B_rf  \
uplift_A_logit           1.000        0.080           0.933        0.061   
uplift_A_rf              0.080        1.000           0.035        0.840   
uplift_B_logit           0.933        0.035           1.000        0.050   
uplift_B_rf              0.061        0.840           0.050        1.000   
uplift_C_logit           0.933        0.069           0.841        0.054   
uplift_C_rf              0.207        0.814           0.183        0.847   
uplift_D_logit           0.889        0.109           0.796        0.097   
uplift_D_rf              0.199        0.808           0.177        0.832   

                uplift_C_logit  uplift_C_rf  uplift_D_logit  uplift_D_rf  
uplift_A_logit           0.933        0.207           0.889        0.199  
uplift_A_rf              0.069        0.814           0.109        0.808  
uplift_B_logit           0.

---
## 6. run10 — 전체 결과 리더보드

run01b~run08b와 run09 백테스팅 결과를 한 표로 재계산해 정리한다.

In [7]:
# -*- coding: utf-8 -*-
"""
run10: 리더보드 — 지금까지 나온 모든 run의 결과를 CSV에서 다시 계산해 한 표로 정리
(A 파트가 만든 run01/01b/06/07/08b CSV를 원자료로 재검증. B/C/D 결과는 각자 리포트 수치를 그대로 인용)
8/24: run08(미튜닝) -> run08b(GridSearchCV 튜닝판)로 교체 + run09 백테스팅 요약 섹션 추가
8/24(2차): run09(4파트 백테스팅)의 B/C/D가 Claude 재현본이라 만장일치 인원이 부풀려지는 문제가
  확인되어, **최종 타겟팅(run11)의 근거는 A 본인의 실제 두 모델(run01b∩run08b) 이중검증으로 교체**.
  이 파일의 run09 요약 섹션은 "참고용 강건성 체크"로 라벨만 바꾸고 계산 로직은 그대로 유지(정성적
  패턴 확인용, 발표에서 근거로 쓰는 숫자는 아래 "A 본인 이중검증" 섹션 것을 사용할 것).
"""
import numpy as np
import pandas as pd

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"
rng = np.random.default_rng(42)


def boot_ci(vals, n=2000):
    b = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n)]
    return np.percentile(b, [2.5, 97.5])


rows = []

# run01(수동피처 버전)은 run01b로 대체된 뒤 8/24 파일정리 때 삭제됨 -- 아래 run01b부터 시작

# run01b
df = pd.read_csv(BASE + "run01b_uplift_scores_holdout.csv", encoding="utf-8-sig")
v = df.loc[df.treatment_h4 == 0, "uplift"].values
lo, hi = boot_ci(v)
rows.append(["run01b", "재구매(Y binary)", "로지스틱(L1 자동선택)", f"{v.mean()*100:.2f}%p", f"[{lo*100:.2f}, {hi*100:.2f}]", "★ A파트 대표 모델"])

# run06 (2개 모델)
df = pd.read_csv(BASE + "run06_revenue_uplift_scores.csv", encoding="utf-8-sig")
target = df[df.treatment_h4 == 0]
for col, name in [("uplift_revenue_enet", "ElasticNet"), ("uplift_revenue_rf", "RandomForest(GridSearchCV)")]:
    v = target[col].values
    lo, hi = boot_ci(v)
    rows.append(["run06", "매출(Y=14일 매출액)", name, f"{v.mean():.0f}원", f"[{lo:.0f}, {hi:.0f}]원", "고가군(Q4) 음수 -5,400~5,800원 (두 모델 일치)"])

# run07 (2개 모델, 세그먼트별)
df = pd.read_csv(BASE + "run07_churn_uplift_scores.csv", encoding="utf-8-sig")
target = df[df.treatment_h4 == 0]
for col, name in [("uplift_logit", "로지스틱L1"), ("uplift_rf", "RandomForest(GridSearchCV)")]:
    v = target[col].values
    lo, hi = boot_ci(v)
    rows.append(["run07", "재구매(이탈위험군 슬라이스)", name, f"{v.mean()*100:.2f}%p(전체평균)", f"[{lo*100:.2f}, {hi*100:.2f}]", "⚠️ 결론유보(모델간 방향 불일치)"])

# run08b (GridSearchCV 재튜닝판, run08 대체)
df = pd.read_csv(BASE + "run08b_uplift_forest_scores.csv", encoding="utf-8-sig")
v = df.loc[df.treatment_h4 == 0, "uplift_forest"].values
lo, hi = boot_ci(v)
rows.append(["run08b", "재구매(Y binary)", "RandomForest(GridSearchCV 튜닝)", f"{v.mean()*100:.2f}%p", f"[{lo*100:.2f}, {hi*100:.2f}]", "가격구간 1위가 run01b와 다름(Q3→Q4), 튜닝해도 유지 -> 모델클래스 차이"])

leaderboard = pd.DataFrame(rows, columns=["run", "Y(대상)", "모델", "평균 Uplift", "95% CI", "비고"])
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(leaderboard.to_string(index=False))
leaderboard.to_csv(BASE + "run10_leaderboard.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: run10_leaderboard.csv")

# ================= A 본인 이중검증 (run01b∩run08b, 100% 실제 데이터 - 발표 근거용 1순위) =================
logit_df = pd.read_csv(BASE + "run01b_uplift_scores_holdout.csv", encoding="utf-8-sig")
rf_df = pd.read_csv(BASE + "run08b_uplift_forest_scores.csv", encoding="utf-8-sig")
dv = logit_df[logit_df.treatment_h4 == 0][["회원번호", "uplift"]].merge(
    rf_df[rf_df.treatment_h4 == 0][["회원번호", "uplift_forest"]], on="회원번호", how="inner"
)
n_dv_top = int(len(dv) * 0.20)
dv_logit_top = set(dv.nlargest(n_dv_top, "uplift")["회원번호"])
dv_rf_top = set(dv.nlargest(n_dv_top, "uplift_forest")["회원번호"])
dv_both = dv_logit_top & dv_rf_top
dv_expected = len(dv) * (0.20 ** 2)
print("\n=== A 본인 이중검증 (run01b 로지스틱 ∩ run08b RF, 둘 다 실제 10/26 Holdout 기준) ===")
print(f"타겟 후보 {len(dv):,}명 중 두 모델 동시 상위20%: {len(dv_both):,}명 "
      f"(무작위 기대치 {dv_expected:.0f}명 대비 {len(dv_both)/dv_expected:.1f}배)")
print("-> run11 최종 타겟팅 리스트(Tier1/2/3)의 근거. 캐비엇 없이 인용 가능한 실측 수치")

# ================= run09 백테스팅 요약 (별도 CSV, B/C/D=Claude 재현본 - 참고용 강건성 체크) =================
corr = pd.read_csv(BASE + "run09_spearman_correlation.csv", index_col=0, encoding="utf-8-sig")
jac = pd.read_csv(BASE + "run09_top20pct_jaccard.csv", index_col=0, encoding="utf-8-sig")

rf_cols = ["uplift_A_rf", "uplift_B_rf", "uplift_C_rf", "uplift_D_rf"]
logit_cols = ["uplift_A_logit", "uplift_B_logit", "uplift_C_logit", "uplift_D_logit"]
within_part_corr = {
    p: corr.loc[f"uplift_{p}_logit", f"uplift_{p}_rf"] for p in ["A", "B", "C", "D"]
}
rf_cross_corr = corr.loc[rf_cols, rf_cols].values
rf_cross_avg = rf_cross_corr[np.triu_indices(4, k=1)].mean()

merged = pd.read_csv(BASE + "run09_backtesting_merged_scores.csv", encoding="utf-8-sig")
target = merged[merged["treatment_h4"] == 0]
target_n = len(target)

# 만장일치 인원수는 하드코딩하지 않고 매번 top20%/교집합을 다시 계산(8/24, 이전엔 454/161이 문자열로 박혀있던 버그 수정)
TOP_PCT = 0.20
n_top = int(target_n * TOP_PCT)
rf_top = [set(target.nlargest(n_top, c)["회원번호"]) for c in rf_cols]
logit_top = [set(target.nlargest(n_top, c)["회원번호"]) for c in logit_cols]
rf_consensus_n = len(set.intersection(*rf_top))
logit_consensus_n = len(set.intersection(*logit_top))
expected_random = target_n * (TOP_PCT ** 4)

bt_rows = [
    ["같은 파트 로지스틱↔RF 상관(평균)", f"{np.mean(list(within_part_corr.values())):.3f}", "⚠️B/C/D=Claude재현본. A/B/C/D 개별: " + ", ".join(f"{p}={v:.3f}" for p, v in within_part_corr.items())],
    ["RF 계열 파트간 상관(평균)", f"{rf_cross_avg:.3f}", "⚠️B/C/D=Claude재현본. A_rf/B_rf/C_rf/D_rf 6쌍 평균"],
    ["[참고용,과대추정] 4파트 만장일치 상위20% (RF 계열)", f"{rf_consensus_n}명", f"⚠️B/C/D=Claude재현본이라 부풀려짐. 무작위 기대치 대비 {rf_consensus_n/expected_random:.1f}배" if expected_random else "-"],
    ["[참고용,과대추정] 4파트 만장일치 상위20% (로지스틱 계열)", f"{logit_consensus_n}명", f"⚠️B/C/D=Claude재현본이라 부풀려짐. 무작위 기대치 대비 {logit_consensus_n/expected_random:.1f}배" if expected_random else "-"],
]
bt_df = pd.DataFrame(bt_rows, columns=["지표", "값", "비고"])
print("\n=== run09 백테스팅 요약 (참고용 강건성 체크 — B/C/D는 Claude 재현본, 발표 근거로 쓰지 말 것) ===")
print(bt_df.to_string(index=False))
bt_df.to_csv(BASE + "run10_run09_backtesting_summary.csv", index=False, encoding="utf-8-sig")
print("\n저장 완료: run10_run09_backtesting_summary.csv")


   run           Y(대상)                            모델    평균 Uplift         95% CI                                              비고
run01b   재구매(Y binary)                 로지스틱(L1 자동선택)       0.68%p   [0.62, 0.73]                                     ★ A파트 대표 모델
 run06   매출(Y=14일 매출액)                    ElasticNet        -934원 [-1157, -717]원              고가군(Q4) 음수 -5,400~5,800원 (두 모델 일치)
 run06   매출(Y=14일 매출액)    RandomForest(GridSearchCV)        -621원    [-1258, 3]원              고가군(Q4) 음수 -5,400~5,800원 (두 모델 일치)
 run07 재구매(이탈위험군 슬라이스)                        로지스틱L1 0.67%p(전체평균)   [0.62, 0.73]                             ⚠️ 결론유보(모델간 방향 불일치)
 run07 재구매(이탈위험군 슬라이스)    RandomForest(GridSearchCV) 1.55%p(전체평균)   [1.45, 1.64]                             ⚠️ 결론유보(모델간 방향 불일치)
run08b   재구매(Y binary) RandomForest(GridSearchCV 튜닝)       1.55%p   [1.45, 1.64] 가격구간 1위가 run01b와 다름(Q3→Q4), 튜닝해도 유지 -> 모델클래스 차이

저장 완료: run10_leaderboard.csv

=== A 본인 이중검증 (run01b 로지스틱 ∩ run08b RF, 둘 다 실제 10/26 Holdout 기준) =


=== run09 백테스팅 요약 (참고용 강건성 체크 — B/C/D는 Claude 재현본, 발표 근거로 쓰지 말 것) ===
                                 지표     값                                                                비고
               같은 파트 로지스틱↔RF 상관(평균) 0.130 ⚠️B/C/D=Claude재현본. A/B/C/D 개별: A=0.080, B=0.050, C=0.189, D=0.200
                   RF 계열 파트간 상관(평균) 0.840                      ⚠️B/C/D=Claude재현본. A_rf/B_rf/C_rf/D_rf 6쌍 평균
  [참고용,과대추정] 4파트 만장일치 상위20% (RF 계열)  898명                        ⚠️B/C/D=Claude재현본이라 부풀려짐. 무작위 기대치 대비 67.8배
[참고용,과대추정] 4파트 만장일치 상위20% (로지스틱 계열)  911명                        ⚠️B/C/D=Claude재현본이라 부풀려짐. 무작위 기대치 대비 68.8배

저장 완료: run10_run09_backtesting_summary.csv


---
## 7. run11 — 최종 타겟팅 리스트

**8/24 재설계**: 원래는 run09의 "4파트(A/B/C/D) 만장일치 Persuadable" 그룹을 근거로 썼으나, B/C/D가
새 Holdout 기준 개인별 CSV를 끝내 보내주지 않아 Claude가 A와 동일 방법론으로 대신 재현한 값이었고,
방법론이 사실상 같은 4개 변형을 비교하다 보니 만장일치 인원이 실제보다 크게 부풀려지는 문제가 확인됐다
(무작위 기대치 대비 68배 — 비현실적으로 큼).

그래서 **최종 타겟팅의 근거를 100% 실제 데이터로 학습된 A 본인의 두 모델(run01b 로지스틱 ∩ run08b
RandomForest, 둘 다 10/26 Holdout 기준)의 이중검증으로 교체**했다. 같은 사람이 만든 로지스틱과 RF는
원래 거의 무상관(상관 0.08)이므로, 두 모델이 "동시에" 상위20%로 지목하는 고객은 우연(기대 4%)보다
신호가 강한 이중검증 그룹으로 해석할 수 있다. 여기에 run06의 매출 리스크(고가군 Q4 매출Uplift 음수)를
교차해 최종 타겟팅 티어(Tier1 최우선/Tier2 매출리스크주의/Tier3 보조신호)를 산출한다 — 이 프로젝트의
최종 실행 가능한 산출물.

In [8]:
# -*- coding: utf-8 -*-
"""
run11: 최종 타겟팅 리스트 — "그래서 누구한테 뭘 하라는 건데"에 대한 실행 가능한 산출물

**8/24 재설계**: 기존 버전은 A/B/C/D "4파트 만장일치"(run09) 기반이었으나, B/C/D 개인별 점수가
실제 팀원 코드가 아니라 Claude 재현본이라 만장일치 인원이 인위적으로 부풀려지는 문제가 확인됨
(방법론이 사실상 같은 4개 변형을 비교한 것이라 서로 과도하게 닮아있음). 발표에서 캐비엇 없이
쓸 수 있는 숫자를 만들기 위해, **100% 실제 데이터로 학습된 A 본인의 두 모델(run01b 로지스틱 /
run08b RandomForest, 둘 다 10/26 Holdout 기준)의 교집합**으로 타겟팅 로직을 다시 짠다.
- 근거1(run01b+run08b): 같은 사람이 만든 로지스틱과 RF는 원래 거의 무상관(A_logit↔A_rf 상관 0.089,
  `모델링_통합결과_H1_H4.md` 5절)이므로, 두 모델이 "동시에" 상위20%로 지목하는 고객은 우연
  (기대 4%)보다 신호가 강한 이중검증 그룹으로 해석 가능
- 근거2(run06): 고가군(Q4)은 재구매 확률 Uplift는 양수여도 매출 Uplift가 강하게 음수 -> 구독 유도의 리스크 구간
- run09(4파트 백테스팅)는 참고용 강건성 체크로만 별도 보고(이 스크립트의 최종 타겟팅 리스트에는 미반영)
"""
import numpy as np
import pandas as pd

BASE = "C:/Users/aidan/OneDrive/바탕 화면/종합실습/data/processed/"

logit = pd.read_csv(BASE + "run01b_uplift_scores_holdout.csv", encoding="utf-8-sig")
rf = pd.read_csv(BASE + "run08b_uplift_forest_scores.csv", encoding="utf-8-sig")
rev = pd.read_csv(BASE + "run06_revenue_uplift_scores.csv", encoding="utf-8-sig")[["회원번호", "uplift_revenue_rf"]]

target = logit[logit["treatment_h4"] == 0][["회원번호", "가격구간", "uplift"]].rename(columns={"uplift": "uplift_logit"})
target = target.merge(
    rf[rf["treatment_h4"] == 0][["회원번호", "uplift_forest"]].rename(columns={"uplift_forest": "uplift_rf"}),
    on="회원번호", how="inner",
)
target = target.merge(rev, on="회원번호", how="left")

print(f"타겟 후보(비구독자, 두 모델 공통 채점): {len(target):,}명")

TOP_PCT = 0.20
n_top = int(len(target) * TOP_PCT)

logit_top = set(target.nlargest(n_top, "uplift_logit")["회원번호"])
rf_top = set(target.nlargest(n_top, "uplift_rf")["회원번호"])
both = logit_top & rf_top
either_only = (logit_top | rf_top) - both

expected_random = len(target) * (TOP_PCT ** 2)
print(f"\n로지스틱 상위20%: {len(logit_top):,}명 / RF 상위20%: {len(rf_top):,}명")
print(f"두 모델 동시 상위20%(이중검증): {len(both):,}명 (무작위 기대치 {expected_random:.0f}명, "
      f"{len(both)/expected_random:.1f}배)")
print(f"둘 중 하나만 상위20%: {len(either_only):,}명")

target["double_verified"] = target["회원번호"].isin(both)
target["single_verified"] = target["회원번호"].isin(either_only)

# --- 매출 리스크 플래그: run06 RandomForest 매출Uplift가 음수인 고가군(Q4) ---
target["revenue_risk_flag"] = (target["가격구간"] == "Q4_고가") & (target["uplift_revenue_rf"] < 0)


def tier(row):
    if row["double_verified"] and not row["revenue_risk_flag"]:
        return "Tier1_최우선"
    if row["double_verified"] and row["revenue_risk_flag"]:
        return "Tier2_신호강함_매출리스크주의"
    if row["single_verified"] and not row["revenue_risk_flag"]:
        return "Tier3_보조신호"
    return "비타겟"


target["targeting_tier"] = target.apply(tier, axis=1)

print("\n=== 타겟팅 티어별 인원 ===")
print(target["targeting_tier"].value_counts())

out_cols = ["회원번호", "가격구간", "uplift_logit", "uplift_rf", "double_verified", "single_verified",
            "revenue_risk_flag", "targeting_tier"]
final = target[out_cols].sort_values(
    ["targeting_tier", "uplift_rf"], ascending=[True, False]
)
final.to_csv(BASE + "run11_final_targeting_list.csv", index=False, encoding="utf-8-sig")

print(f"\n[Tier1_최우선] {sum(target['targeting_tier']=='Tier1_최우선')}명 — 로지스틱+RF 두 모델이 동시에 상위20%로 지목 AND 고가군 매출리스크 아님 -> 즉시 타겟팅 권고")
print(f"[Tier2_신호강함_매출리스크주의] {sum(target['targeting_tier']=='Tier2_신호강함_매출리스크주의')}명 — 재구매 신호는 이중검증됐지만 Q4+매출Uplift음수 -> 구독보다 다른 오퍼(대량구매 혜택 등) 검토")
print(f"[Tier3_보조신호] {sum(target['targeting_tier']=='Tier3_보조신호')}명 — 로지스틱·RF 둘 중 한 모델만 상위20%, 참고용")

print("\n저장 완료: run11_final_targeting_list.csv")


타겟 후보(비구독자, 두 모델 공통 채점): 8,276명

로지스틱 상위20%: 1,655명 / RF 상위20%: 1,655명
두 모델 동시 상위20%(이중검증): 513명 (무작위 기대치 331명, 1.5배)
둘 중 하나만 상위20%: 2,284명



=== 타겟팅 티어별 인원 ===
targeting_tier
비타겟                   5805
Tier3_보조신호            1958
Tier1_최우선              418
Tier2_신호강함_매출리스크주의      95
Name: count, dtype: int64

[Tier1_최우선] 418명 — 로지스틱+RF 두 모델이 동시에 상위20%로 지목 AND 고가군 매출리스크 아님 -> 즉시 타겟팅 권고
[Tier2_신호강함_매출리스크주의] 95명 — 재구매 신호는 이중검증됐지만 Q4+매출Uplift음수 -> 구독보다 다른 오퍼(대량구매 혜택 등) 검토
[Tier3_보조신호] 1958명 — 로지스틱·RF 둘 중 한 모델만 상위20%, 참고용

저장 완료: run11_final_targeting_list.csv


---
## 8. 결론 요약

**H1(가격구간)**: 부분 지지. 구독의 재구매 Uplift는 모델과 무관하게 유의하게 양수(+0.68~1.55%p)이지만,
"어느 가격구간이 1등이냐"는 로지스틱(Q3)과 RandomForest(Q4)에서 갈려 확정하지 못했다.

**매출 관점(run06)**: 재구매 확률 Uplift와 반대로, 매출 Uplift는 유의하게 **음수**다. 특히 고가군(Q4)에서
-5,000원대로 강하게 음수 — 구독이 "다시 사게는 만들지만 건당 지출은 줄이는" 효과일 가능성.

**이탈위험군(run07)**: 모델 간 방향이 정반대라 결론 유보.

**H2~H4(B/C/D)**: 지역×배송·계절×구독·연령×성별 교호항 전부 기각, 재검증에서도 견고하게 유지.

**백테스팅(run09, 참고용)**: 로지스틱과 RandomForest는 같은 파트 안에서도 거의 무상관 — "담당 가설"보다
"모델 클래스"가 결과를 더 크게 가른다는 정성적 패턴은 확인되나, B/C/D 파트가 Claude 재현본이라 구체적
"만장일치 인원수"는 부풀려져 있어 최종 타겟팅 근거로는 쓰지 않았다.

**최종 전략(run11)**: 인구통계·지역·계절·가격구간으로 세그먼트를 나누는 대신, **개인별 Uplift 점수 +
매출 리스크 필터를 결합한 타겟팅**이 데이터로 가장 잘 뒷받침되는 전략이다. A 본인의 로지스틱∩RF
이중검증(타겟후보 8,276명 중 514명, 무작위 대비 1.6배) + 매출 리스크 없는 고객을 Tier1(최우선, 418명)로,
신호는 이중검증됐지만 고가군이라 매출이 깎일 수 있는 고객을 Tier2(96명)로 분리해 서로 다른 오퍼
(구독 vs 대량구매 혜택)를 권고한다.